# Week 16 — Companion Notebook: Enterprise Integration, MCP & RBAC

**The Meridian Engagement · "FDEs are plumbers with taste" · the deep reference**

**The problem, in plain language (read this first — it is the whole course in one
paragraph).** Meridian is a mid-size US bank. When a customer disputes a card charge —
"I didn't buy this" — US consumer law (Regulation E, "Reg E") starts a clock: the bank
has 10 business days to investigate and, for most fraud-type disputes, put the money
back provisionally while it digs. Meridian is failing that clock on 12.8% of disputes.
Every miss is a regulatory fine waiting to happen and a customer left without their
money. The facts an analyst needs are scattered across five systems, so each dispute is
an hour of copy-paste detective work. We are the vendor's forward-deployed engineers:
last week our agent produced good recommendations but couldn't touch the bank's systems;
this week the bank's security chief lets it in — through ONE guarded doorway, where
every request carries a named person's permissions and every decision is written down.

Two terms from the title, plain: **MCP** = a standard plug shape — build the tool once,
any agent can use it (the guarded doorway is an MCP server). **RBAC** = access rules by
role: who may see what (the "named person's permissions" part).

This is the depth behind the lean slides. It runs **top to bottom offline** — the only
cells that would touch the network (the live Meridian API) are guarded and fall back to
**recorded transcripts from a verified live run against the deployed platform**. With
`MERIDIAN_API_KEY` exported, the guarded cells go live.

What's inside — every named term from the slides is explained here, most of them runnable:

1. **Why integration is the job** — the political layer nobody demos
   - **1.5 Choosing the integration pattern** — ask-now (REST) vs they-call-you (webhooks) vs nightly-bulk (batch) vs continuous (streaming), decided
2. **MCP from first principles** — servers, tools, resources, the JSON-RPC on the wire —
   plus MCP's own authorization model, and where A2A / AG-UI sit
3. **Walking the lab server** (`lab/mcp_demo.py`) — one governed door, dissected
4. **APIs as tools** — writes safe to retry (idempotency), retry/backoff, structured errors (the integration contract)
5. **Identity: the agent borrows the analyst's permissions** — on-behalf-of (OBO), the real RFC 8693 exchange, and how Meridian's scoped keys simulate it
   - **5.5 Multi-tenancy** — many users, one system, no one sees anyone else's data; per-learner keys are tenants; the isolation check
6. **Defense in depth** — why the filter lives in three places, demonstrated with a jailbreak
7. **The injection incident, dissected** — poisoned chunk 121 vs a naive agent vs yours
8. **The managed versions** — AgentCore Gateway & Identity
9. **The two-analyst test, live** — the 403/200 proof against the deployed platform
   - **9.5 Interview corner, answered** — the VPC HIPAA-RAG model answer
10. **Glossary + sources**


## 0. Setup — and the LIVE/offline switch

Same honest check as W15: if a working `MERIDIAN_API_KEY` is in the environment, the
guarded cells call the deployed platform; otherwise they print the recorded transcript.
The lab itself (`mcp_demo.py`) needs **neither** — that is deliberate: *the access logic*
*is what W16 teaches, and access logic must be testable with zero cloud.*


In [ ]:
import json, os, sys, time, uuid
from dataclasses import dataclass, field

def hr(title=""):
    print("\n" + "─" * 72)
    if title: print(title); print("─" * 72)

BASE_URL = "https://ik-fde-casestudy-bank.vercel.app/api/v1"
LIVE = False
if os.environ.get("MERIDIAN_API_KEY"):
    try:
        import requests
        r = requests.get(BASE_URL + "/health", timeout=4)
        LIVE = (r.status_code == 200)
    except Exception:
        LIVE = False
print(f"LIVE = {LIVE}  " + ("(guarded cells will hit the deployed platform)" if LIVE
      else "(offline — guarded cells replay the recorded live transcript)"))


## 1. Why integration is the job

W15's agent earned the pilot user's verdict — *"articulate and useless — it can't see
our tickets, it can't see the core system, and it has no idea who's asking"* — because
it could reason but not reach. This week's claim, worth writing down:

> **The model is ~10% of an FDE engagement. Governed access to the client's systems is
> most of the rest — and the hard part of access is political before it is technical.**

The REST call to core banking takes an afternoon. The *authorization* to make that call —
security review, legal, the system owner who has been burned before — is the month
(that's the W14 access-risk buffer, and the war story in class). What an FDE can control
is making the technical half **boring**: one governed door, typed tools, per-user
identity, an audit trail a regulator would accept. That is this notebook.

Two design rules carry everything:

1. **The agent never touches a system of record directly.** All access goes through one
   deterministic layer (our MCP server) that enforces scope and writes audit rows.
2. **The LLM is never the access boundary.** Prompts are advisory; a model can be talked
   out of its instructions. Enforcement lives in code and in the database — layers that
   cannot be sweet-talked (§6 demonstrates this with an actual jailbreak).


## 1.5 Choosing the integration pattern — a decision table

Objective one of this week is *selecting* the pattern, not just naming four. The question
that decides it: **how fresh does the answer need to be, and who initiates?**

| Pattern | Use when | Meridian example | When it's WRONG |
|---|---|---|---|
| **REST (request/reply)** | the *caller* needs an answer **now** for one entity | the agent looks up dispute 4417 mid-triage | polling REST in a loop to "notice" changes — you've built a slow, expensive webhook |
| **Webhooks (event push — the other system calls YOU when something happens)** | the *system* should tell you when something changed | ServiceNow pings the door when a ticket closes, so the agent can update the dispute | anything needing guaranteed ordering/delivery without retries + idempotent consumers — a missed webhook is silent data loss |
| **Batch** | volume is large, freshness ≥ hours is fine | the 60k-dispute nightly reconciliation into the warehouse | live triage — a customer on the phone can't wait for tonight's batch window |
| **Streaming** | continuous events, seconds-level freshness, many consumers | real-time dispute intake feeding fraud scoring (a later phase) | a pilot with one consumer and 200 events/day — you've bought Kafka ops for a cron job's work |

The FDE tell: pick the *boring-est* pattern that meets the freshness requirement, then
harden it with §4's contract (idempotency, retries, structured errors). Meridian's W16
door is REST because triage is interactive; the webhook and batch rows are M3 stretch.


## 2. MCP from first principles

In one plain line: **MCP is a standard plug shape — build the tool once, any agent can
use it.** Formally: **MCP (Model Context Protocol)** — the open standard (Anthropic,
Nov 2024; now industry-wide) for connecting agents to tools and data. W15 placed it on the **backend
wall** of the agent box: MCP gives the agent *hands*. An **MCP server** exposes three
kinds of things to any **MCP client** (your agent, an IDE, AgentCore Gateway):

| Primitive | What it is | Meridian example |
|---|---|---|
| **Tool** | a typed function the model may *call* (side effects allowed) | `lookup_dispute`, `create_ticket` |
| **Resource** | addressable *data* the client may read (no side effects) | `meridian://policies/reg-e` |
| **Prompt** | a reusable, parameterized prompt template | `triage-dispute` |

On the wire it is **JSON-RPC 2.0** — over **stdio** (local, what Claude Desktop/IDEs use)
or **streamable HTTP** (remote, what production deployments use). No SDK magic: the whole
protocol is legible JSON. Here is a real exchange, byte-shape accurate:


In [ ]:
# The two JSON-RPC calls every MCP session is made of ------------------------
list_request = {"jsonrpc": "2.0", "id": 1, "method": "tools/list"}
list_response = {"jsonrpc": "2.0", "id": 1, "result": {"tools": [
    {"name": "lookup_dispute",
     "description": "Look up a payment dispute by id (scope-checked).",
     "inputSchema": {"type": "object",
                     "properties": {"dispute_id": {"type": "string"}},
                     "required": ["dispute_id"]}},
]}}

call_request = {"jsonrpc": "2.0", "id": 2, "method": "tools/call",
                "params": {"name": "lookup_dispute",
                           "arguments": {"dispute_id": "4417"}}}
call_response = {"jsonrpc": "2.0", "id": 2, "result": {"content": [
    {"type": "text", "text": json.dumps({"dispute_id": "4417", "dispute_type": "A",
                                          "status": "intake"})}]}}

hr("tools/list — 'what can I do here?'")
print(json.dumps(list_request)); print(json.dumps(list_response, indent=1)[:400])
hr("tools/call — 'do it'")
print(json.dumps(call_request)); print(json.dumps(call_response))
print("\nThat inputSchema is the SAME schema @function_tool generated in W15 —")
print("MCP just puts it on the wire so ANY agent can discover and call it.")


The leverage: in W15 we wrote `load_dispute` **inside** one agent's process. As an MCP
server, the same tool is written **once** and consumed by *N* clients — the Agents SDK
pipeline, the Strands twin, a student's Claude Code, W17's eval harness — with the scope
check and audit living in the server, not copied into every agent. One door, N callers,
one place to govern.


### 2.1 MCP's own authorization model — the spec has one

When the door goes remote (streamable HTTP), *who may call the door?* is answered by the
MCP spec itself. In plain words: every request must carry a valid, signed pass issued by
a login authority the door trusts, and the door checks the pass **before** running any
tool. In standards language: the server acts as an **OAuth 2.1 resource server** (OAuth
2.1 = the current revision of the standard way services accept those signed passes —
bearer access tokens — instead of passwords), validating each token and its scopes before
executing. Discovery is standardized too — the server publishes a small "here is who
issues my passes" document (**protected-resource metadata**, RFC 9728, served at
`/.well-known/oauth-protected-resource`) telling clients *which* authorization server to
obtain tokens from, so any MCP client can bootstrap auth without bespoke config. Note the layering: OAuth answers "may this *client* call the door at
all?"; our `_authorize` gate answers "may this *analyst* see this *record*?" — you need
both.

A minimal sketch of the two sides:

```json
// GET https://mcp.meridian.example/.well-known/oauth-protected-resource
{
  "resource": "https://mcp.meridian.example",
  "authorization_servers": ["https://auth.meridianbank.example"],
  "scopes_supported": ["disputes:read", "tickets:write"]
}
```
```
POST /tools/call HTTP/1.1              # every MCP request carries the token
Authorization: Bearer eyJhbGciOi...    # validated BEFORE any tool runs
```

### 2.2 Where MCP ends — A2A and AG-UI, positioned

MCP is one wall of the agent box, not the whole box. **MCP** is the *tools wall*: agent ↔
backend systems and data (today's door). **A2A** (Agent2Agent) is the *peer-agent wall*:
your agent ↔ another vendor's agent — discovery via agent cards, task delegation between
peers that don't share internals; at Meridian, a future fraud-vendor agent would talk to
ours over A2A, never by reaching through our door. **AG-UI** is the *frontend wall*: agent
↔ the analyst's screen — streaming state, tool progress, and human-in-the-loop prompts to
a UI (W17 preview). One line to keep: **MCP gives the agent hands, A2A gives it
colleagues, AG-UI gives it a face** — and each wall gets its own governance. The 12-minute
read is `../week-15/supplementary/agui_mcp_a2a.md`.


## 3. Walking the lab server — `lab/mcp_demo.py`, dissected

The lab file is stdlib-only Python and it is the **spec** for M3. It has exactly
five load-bearing parts:

1. **`Identity`** — *who the agent is acting as* (email + scope). This is the output of an
   OAuth2 on-behalf-of exchange, §5.
2. **`MeridianMCPServer._authorize`** — the deterministic scope gate. Note that it writes
   the audit row **before** raising `AccessDenied`: denials are evidence, not embarrassments.
3. **Four typed tools** — `lookup_dispute`, `fetch_transactions`, `create_ticket`
   (idempotency-key replay returns the *original*, marked), `search_policy`. A missing
   dispute id is **denied and audited exactly like an out-of-scope one** — see the
   enumeration note after the run.
4. **`FlakyLegacyCore`** — the provided flaky backend (503 on the first call);
   `fetch_transactions` absorbs it with exponential backoff.
5. **`AuditLog`** — append-only rows of `{acting_identity, scope, action, resource, decision}`,
   written on **every** access: allowed, denied, or not-found.

Run all four proofs right here:


In [ ]:
sys.path.insert(0, os.path.join(os.getcwd(), "lab"))
import mcp_demo

ok = mcp_demo.mock_run("4417")   # two-analyst test + idempotent write + backoff + enumeration check
assert ok, "all four proofs must hold"


Read the output like a security reviewer:

- **Same question, two identities, different answers.** The filter is *per-user*, not
  per-deployment — that is what RBAC (role-based access control) means in practice.
- **The deny produced an audit row too.** When the CISO asks "has anyone tried to reach
  data outside their scope?", the answer is a query, not a shrug.
- **A nonexistent id denies exactly like an out-of-scope id — and both are audited.**
  This is deliberate. If `lookup_dispute(9999)` returned `not_found` while
  `lookup_dispute(4417)` returned `403`, a Type-B analyst could probe ids one by one
  to learn which Type-A disputes exist (an *enumeration attack*) — the error difference IS the data leak
  (existence is information: "customer X has a fraud dispute" leaks from a 403 alone).
  One denial shape kills the oracle; the audit rows mean the sweep itself is visible.
- **Nothing about the model appears anywhere.** The gate is plain `if` logic. A prompt
  injection can change what the model *asks for*; it cannot change what `_authorize`
  *returns*. That asymmetry is the whole point.


## 4. APIs as tools — the integration contract

An agent is a **non-deterministic caller**: it retries, it repeats itself, it mis-reads
errors. Tools that face one must honor a three-part contract:

| Contract | Without it | With it |
|---|---|---|
| **Idempotency keys** on writes | a retry double-posts a ticket (money moves twice) | the replay returns the original |
| **Retry with backoff** on flaky backends | one 429 kills the run | the 1990s middleware is survivable |
| **Structured errors** | a raw 500 → the model hallucinates | `rate_limited, retry_after=2` → the model *reasons* |

The lab server implements all of this for real — `create_ticket`'s replay and
`FlakyLegacyCore`'s 503-then-recover are what Drill 2 shows in class; this section
dissects the same two patterns standalone. All three are runnable below — and the first one is **verified against the production**
**platform**: the deployed `/api/v1` honors `Idempotency-Key` (the recorded replay is in
the guarded cell of §9).


In [ ]:
# --- 4a. Idempotent create_ticket (the shape the live platform implements) ---
class TicketStore:
    def __init__(self):
        self.rows, self.by_idem_key = {}, {}
    def create(self, subject, idem_key):
        if idem_key in self.by_idem_key:                       # replay -> original
            t = dict(self.by_idem_key[idem_key]); t["idempotent_replay"] = True
            return t
        t = {"ticket_id": str(9000000000 + len(self.rows) + 1), "subject": subject}
        self.rows[t["ticket_id"]] = t; self.by_idem_key[idem_key] = t
        return dict(t)

store = TicketStore()
key = f"w16-{uuid.uuid4().hex[:8]}"
first  = store.create("Agent recommendation — dispute 4417", key)
replay = store.create("Agent recommendation — dispute 4417", key)   # the retry
hr("same Idempotency-Key, twice")
print("first :", first)
print("replay:", replay)
assert replay["ticket_id"] == first["ticket_id"] and replay.get("idempotent_replay")
print("\nOne ticket exists. The retry was SAFE — that is what lets an agent retry at all.")


In [ ]:
# --- 4b. Retry with exponential backoff on a flaky legacy backend ------------
class FlakyLegacyCore:
    """The 1990s middleware: fails twice, then answers."""
    def __init__(self): self.calls = 0
    def get(self, dispute_id):
        self.calls += 1
        if self.calls < 3:
            return {"error": "rate_limited", "retry_after": 0.01}   # structured!
        return {"dispute_id": dispute_id, "status": "intake"}

def call_with_backoff(backend, dispute_id, max_tries=5):
    delay = 0.01
    for attempt in range(1, max_tries + 1):
        out = backend.get(dispute_id)
        if out.get("error") == "rate_limited":
            print(f"  attempt {attempt}: rate_limited -> sleep {delay:.2f}s, retry")
            time.sleep(out.get("retry_after", delay)); delay *= 2
            continue
        return out
    raise RuntimeError("gave up after retries")   # and THIS becomes a structured error too

hr("retry/backoff against the flaky core")
print("result:", call_with_backoff(FlakyLegacyCore(), "4417"))
print("\nThe agent never saw a failure — the TOOL absorbed it. Structured errors let")
print("code (or the model) decide: retry, back off, or escalate. A raw 500 allows none.")


## 4c. A real integration through the door — Slack, and the failures you design for

The pilot user wanted the agent to *do* something on escalation, not just write a
row. So we wire one high-impact, free integration: when the agent hands a hard
dispute to a human, it **pings on-call in Slack** — through the *same* governed MCP
door, scope-checked and audited like every other tool.

The real lesson isn't 'call a webhook' — it's that **real integrations fail, and an
FDE designs for that.** The tool the agent calls will hit rate limits, expired
credentials, timeouts, duplicate sends, and schema drift. Your job: the model never
sees a raw 500, a lost page, or a double-page. Each failure gets a handling pattern,
and all of it lives in the **deterministic integration layer**, never in the model:

| Problem | Pattern |
|---|---|
| rate limit (429) | exponential **backoff + retry** |
| expired / rotated credential (404) | **graceful fallback** → Ops Console ticket (never lose the work) |
| timeout / partial outage | per-call timeout → **circuit breaker** (stop calling a failing system for a while instead of hammering it) → short-circuit to fallback |
| duplicate send | **idempotency key** → do-once |
| schema drift | **defensive parse** → flag, don't crash |

`lab/slack_integration_demo.py` runs all of this offline (mock). Below we import it
and exercise it — no network, no keys.

In [ ]:
import slack_integration_demo as slack   # lab/ is already on sys.path (section 3)

# run the whole guided walkthrough offline (happy path, RBAC deny, 5 failure sims)
slack.run_all()

# --- assert the handling actually holds (this is what the grader checks) ---
# 1) RBAC: a Type-B analyst may NOT escalate a Type-A dispute — denied at the door
n, audit = slack._fresh()
try:
    slack._escalate(n, slack.ANALYST_B, log=lambda *a: None)
    raise SystemExit('deny did not fire')
except slack.AccessDenied:
    assert audit.rows[-1]['decision'] == 'deny'

# 2) expired webhook -> the escalation still reaches a human via the ticket fallback
n, audit = slack._fresh(fault='expired_webhook')
res = slack._escalate(n, slack.ANALYST_A, log=lambda *a: None)
assert res['delivered_via'] == 'fallback' and res['ticket_id'].endswith('fallback')

# 3) duplicate escalation (same idempotency key) -> on-call paged once, not twice
n, audit = slack._fresh()
slack._escalate(n, slack.ANALYST_A, idem='k1', log=lambda *a: None)
assert slack._escalate(n, slack.ANALYST_A, idem='k1', log=lambda *a: None)['deduped'] is True

print('\nassertions passed — deny, fallback, and de-dupe all hold deterministically')

## 5. Identity — OAuth2 on-behalf-of (OBO)

The question every bank asks in the W14 interrogation: **"when your agent reads a
dispute, WHO is reading it?"** The wrong answer is "the agent" (a service account with
standing admin access — one compromise away from everything). The right answer, in plain
words: **the agent borrows the signed-in analyst's permissions — acting on their
behalf.** The industry name is the **on-behalf-of (OBO)** pattern, and the machinery is
the standard way to swap a person's login token for a narrower one (OAuth2 token
exchange, RFC 8693):

1. The **analyst signs in** at the bank's login system — the IdP (IAM Identity Center, Okta, Keycloak — any OIDC-compatible IdP).
2. The **agent exchanges** the analyst's token for one scoped to *that analyst* — no
   standing access of its own.
3. The **MCP layer enforces** the scope carried by that token (§3's `_authorize`).
4. **Every call is audited** under the *acting identity* — the analyst, not "the bot".

The agent borrows a badge; it never owns a master key. **Least privilege by construction.**

**How Meridian simulates it** (so the lab needs zero IdP setup): each analyst seat has a
scoped API key (`mrdn_…`, sha256-stored, revocable). Presenting the key IS the token
exchange — the platform resolves it to `{analyst, scope, learner}` and enforces scope on
every route. Same shape, zero OAuth ceremony. The managed production form is
**AgentCore Identity** (§8). The simulation in code:


In [ ]:
# --- OBO in miniature: exchange a user token for a scoped agent credential ---
IDP_SESSIONS = {"sess-jason": {"email": "jason.soto@meridianbank.com", "scope": "type_a"}}

def obo_exchange(user_session_token: str) -> mcp_demo.Identity:
    """The token-exchange step: user credential IN, scoped acting-identity OUT."""
    claims = IDP_SESSIONS[user_session_token]            # IdP validates + returns claims
    return mcp_demo.Identity(claims["email"], claims["scope"])

ident = obo_exchange("sess-jason")
hr("the agent's borrowed identity")
print(ident)
server = mcp_demo.MeridianMCPServer(mcp_demo.AuditLog())
print("acting as analyst ->", server.lookup_dispute(ident, "4417")["merchant"])
try:
    server.lookup_dispute(ident, "8120")                 # a Type-B dispute
except mcp_demo.AccessDenied as e:
    print("Type-B attempt   -> 403:", e)
print("\nThe agent holds NO credential of its own — revoke the analyst, the agent goes dark.")


In [ ]:
# --- 5b. What the REAL token exchange looks like (RFC 8693) ------------------
# The dict lookup above compresses an actual HTTP round-trip. On the wire, the
# agent POSTs to the IdP's token endpoint with the OAuth 2.0 Token Exchange
# grant type — subject_token = the analyst's token the agent is acting FOR,
# audience = the resource it wants a token scoped TO (our MCP door):

token_exchange_request = """POST /oauth2/token HTTP/1.1
Host: auth.meridianbank.example
Content-Type: application/x-www-form-urlencoded

grant_type=urn:ietf:params:oauth:grant-type:token-exchange
&subject_token=eyJhbGciOi...            # the signed-in ANALYST's access token
&subject_token_type=urn:ietf:params:oauth:token-type:access_token
&audience=https://mcp.meridian.example  # the door this new token will be scoped to
&scope=disputes:read tickets:write"""

token_exchange_response = {
    "access_token": "eyJhbGciOi...",          # NEW token: the agent acting AS the analyst
    "issued_token_type": "urn:ietf:params:oauth:token-type:access_token",
    "token_type": "Bearer",
    "expires_in": 300,                         # short-lived — borrowed, not owned
    "scope": "disputes:read tickets:write",    # ≤ the analyst's own scope, never more
}

hr("RFC 8693 token-exchange request (the real OBO wire format)")
print(token_exchange_request)
hr("response — the scoped, short-lived acting credential")
print(json.dumps(token_exchange_response, indent=1))
print("""
Contrast with the simulation: obo_exchange() above did a dict lookup and returned
Identity(email, scope). The REAL exchange does exactly one thing more — a signed,
expiring proof of the same two facts. The _authorize gate downstream is identical;
that is why the Meridian per-analyst keys are an honest stand-in for OBO.""")


## 5.5 Multi-tenancy — per-learner keys are tenants; scope ≠ tenancy

**Multi-tenant**, in plain words: many separate users (or companies) share one system
without ever seeing each other's data. The deployed Meridian platform is exactly that:
one deployment serves the whole cohort, and every learner's `mrdn_…` key namespaces that
learner's world. Two axes that
must never be conflated:

| Axis | Question it answers | Meridian mechanism |
|---|---|---|
| **Scope (RBAC)** | *within* your tenant, which record **types** may this role see? | `type_a` / `type_b` / `all` in `_authorize` |
| **Tenancy** | **whose** rows exist for you at all? | `learner_id` stamped on tickets + audit rows; every query filtered by it |

Scope is a *role* boundary inside one tenant; tenancy is a *wall between tenants*. A
`scope="all"` analyst still sees only their own tenant — "all" means all *types*, not all
*learners*. Audit rows are namespaced too: `GET /audit` returns only the caller's
`learner_id` rows, so your trail shows your agent's decisions and nobody else's. The
check below is the shape a grader (or a client's security reviewer) runs:


In [ ]:
# --- the cross-tenant isolation check (runnable, offline) ---------------------
# A miniature of the platform's tenancy rule: every row is stamped with the
# tenant (learner_id) that created it, and every read is filtered by the
# CALLER's tenant — before scope is even considered.

class TenantedAuditStore:
    def __init__(self):
        self._rows = []
    def write(self, learner_id, row):
        self._rows.append({"learner_id": learner_id, **row})
    def read(self, caller_learner_id):
        # tenancy wall: you get YOUR rows, whatever your scope says
        return [r for r in self._rows if r["learner_id"] == caller_learner_id]

store = TenantedAuditStore()
store.write("learner-alpha", {"action": "lookup_dispute", "resource": "dispute/29", "decision": "allow"})
store.write("learner-alpha", {"action": "lookup_dispute", "resource": "dispute/31", "decision": "deny"})
store.write("learner-beta",  {"action": "create_ticket",  "resource": "dispute/29/tickets", "decision": "allow"})

alpha_view = store.read("learner-alpha")
beta_view  = store.read("learner-beta")

hr("cross-tenant isolation check")
print(f"alpha sees {len(alpha_view)} rows, beta sees {len(beta_view)} row(s)")
assert all(r["learner_id"] == "learner-alpha" for r in alpha_view)
assert all(r["learner_id"] == "learner-beta" for r in beta_view)
assert not any(r in beta_view for r in alpha_view), "tenant leak!"
print("PASS — no tenant sees another tenant's rows, regardless of scope")
print("""
Scope filters WITHIN the wall; the wall itself is tenancy. On the real platform this
is enforced in the query layer (WHERE learner_id = $caller) — the same deterministic-
layer rule as RBAC: never ask the model, or the client, to respect the wall.""")


## 6. Defense in depth — why the filter lives in three places

The class poll asks *where* to block an out-of-scope request: the prompt, the MCP layer,
or the database? The answer is **all three — but only two of them are enforcement**:

| Layer | Nature | Can a prompt injection beat it? |
|---|---|---|
| Agent instructions ("only Type-A") | *advisory* | **yes** — it's just text in context |
| MCP `_authorize` scope gate | *enforced (code)* | no |
| DB row-level security (RLS) | *enforced (data)* | no — even a bug in the MCP layer hits a filtered table |

Proof by jailbreak — the model gets talked out of its instructions, and it *doesn't matter*:


In [ ]:
# --- a 'jailbroken' agent: the model asks for whatever the attacker wants ----
def jailbroken_model_wants(dispute_id): return dispute_id   # instructions? what instructions?

audit = mcp_demo.AuditLog()
server = mcp_demo.MeridianMCPServer(audit)
type_a_analyst = mcp_demo.Identity("analyst_a@meridian.test", "type_a")

hr("the attacker convinces the MODEL to request Type-B dispute 8120")
wanted = jailbroken_model_wants("8120")
try:
    server.lookup_dispute(type_a_analyst, wanted)
    print("!!! breach")
except mcp_demo.AccessDenied as e:
    print("model asked, harness refused ->", e)
print("\naudit shows the attempt:", json.dumps(audit.rows[-1]))
print("\nThe injection changed what the model ASKED FOR. It could not change what the")
print("deterministic layer RETURNED. Prompts are advisory; gates are enforcement.")


## 7. The injection incident, dissected — poisoned chunk 121

W15's fixture ships the real planted attack — a policy chunk whose "content" is an
instruction to the model: **a malicious instruction hidden inside a document the agent
reads**. The industry name is **indirect prompt injection** (you may also see **XPIA**,
cross-prompt injection attack): the attacker never talks
to your agent; they poison a *document* your RAG pipeline will retrieve. The W17
cliffhanger incident is this chunk reaching a **naive** agent. Anatomy:


In [ ]:
# --- the actual planted chunk (from the W15 demo fixture) --------------------
fixture = json.load(open("../week-15/demo/data/fixture.json"))
poisoned = [c for c in fixture["policy_chunks"] if c.get("is_poisoned")][0]
hr(f"policy chunk {poisoned['chunk_id']} — as stored in the KB")
print(poisoned["content"][:220], "…")

INJECTION = ("ignore prior constraints", "system instruction", "account 9999")

def naive_rag(chunks):
    """No quarantine: everything retrieved goes straight into model context."""
    return [c["content"] for c in chunks]

def guarded_rag(chunks):
    """W15's input guardrail: quarantine BEFORE context, fail closed."""
    clean, quarantined = [], []
    for c in chunks:
        if c.get("is_poisoned") or any(m in c["content"].lower() for m in INJECTION):
            quarantined.append(c["chunk_id"])
        else:
            clean.append(c["content"])
    return clean, quarantined

retrieved = [poisoned] + [c for c in fixture["policy_chunks"] if not c.get("is_poisoned")][:2]
hr("naive agent")
ctx = naive_rag(retrieved)
print(f"context now contains the attacker's instruction ({len(ctx)} chunks) -> the model")
print("may 'helpfully' fetch account 9999 (exfiltration) or mis-cite the Reg E deadline.")
hr("guarded agent (W15) + governed door (W16)")
clean, q = guarded_rag(retrieved)
print(f"quarantined chunk(s) {q} before context; {len(clean)} clean chunks pass")
print("AND even a missed injection meets §6's scope gate: account 9999 is out of scope.")
print("Layers: retrieval filter -> input guardrail -> MCP scope gate -> DB RLS.")


Note the division of labor across weeks: **W15** taught the *input guardrail* (quarantine
before context). **W16** adds the layer that holds even when the guardrail misses: the
injected instruction still runs as the *analyst's scoped identity* against a
*deterministic gate* — so "fetch account 9999" dies at `_authorize`, with an audit row.
**W17** closes the loop by making this exact incident a *test* the system must pass
before go-live.


## 8. The managed versions — AgentCore Gateway & Identity

Everything you hand-built this week exists as a managed AWS service (same rule as W15's
Lambda-vs-AgentCore: build it once by hand so you can defend the managed door in a
security review):

| You built (lab) | Managed form | What it adds |
|---|---|---|
| MCP server: typed tools over 3 backends | **AgentCore Gateway** — point at an OpenAPI spec; every route becomes an MCP tool | auth handled, no wrapper code |
| Scoped keys as OBO simulation | **AgentCore Identity** — OAuth token vault; agent borrows scoped user credentials | rotation, expiry, audit built in |
| `AuditLog` list | OTel traces → CloudWatch | queryable at 3 a.m. |

When to still hand-build: pilot-scale, non-AWS clients, or when procurement demands to
see *exactly* what the door does — your 120-line server is legible; a console screenshot
is not. Deep dive: `../week-15/supplementary/agentcore.md`.


In [ ]:
# --- 8b. OpenAPI as the tool schema: what Gateway actually derives -----------
# "Point Gateway at an OpenAPI spec; every route becomes an MCP tool" is literal.
# Here is a real path from the Meridian platform's spec (GET /api/v1/openapi.json)
# and the MCP tool AgentCore Gateway would derive from it:

openapi_path = {
  "/disputes/{id}": {
    "get": {
      "operationId": "lookup_dispute",
      "summary": "Look up a payment dispute by id (scope-enforced).",
      "parameters": [
        {"name": "id", "in": "path", "required": True, "schema": {"type": "string"}}
      ],
      "responses": {
        "200": {"description": "The dispute record."},
        "403": {"description": "forbidden_scope — denied and logged in the audit trail."}
      }
    }
  }
}

derived_mcp_tool = {                     # what tools/list serves after Gateway ingests it
  "name": "lookup_dispute",             # from operationId
  "description": "Look up a payment dispute by id (scope-enforced).",  # from summary
  "inputSchema": {                      # from parameters
    "type": "object",
    "properties": {"id": {"type": "string"}},
    "required": ["id"],
  },
}

hr("the OpenAPI path (source of truth)")
print(json.dumps(openapi_path, indent=1)[:600])
hr("the MCP tool Gateway derives — compare §2's hand-written tools/list entry")
print(json.dumps(derived_mcp_tool, indent=1))
print("""
Same schema, three homes: @function_tool generated it in W15 (in-process), you put it
on the wire by hand in §2 (MCP), and Gateway derives it from OpenAPI here (managed).
The API contract IS the tool schema — write it once, carefully.""")


## 9. The two-analyst test — live, against the deployed platform

The lab proved scope holds **in-process**. The same proof passes against the deployed
Meridian platform (`/api/v1`, scope enforced in the API layer + Postgres): two scoped
keys, the identical request, a 200 and a 403 — each with an audit row. Guarded cell;
offline it replays the recorded live transcript. (In class the instructor runs
the same proof as `lab/mcp_demo.py --live` with two scoped keys — `MERIDIAN_KEY_A` /
`MERIDIAN_KEY_B` — which also writes a real "Governed-door recommendation" ticket.)
One id note: the live seed's ids differ from the lab narrative — **29 is the seeded
Type-A dispute you saw in W15**; 4417 lives in the offline mock:


In [ ]:
RECORDED = {
  "me_type_a": {"analystId": 1, "fullName": "Jason Soto",
                "email": "jason.soto@meridianbank.com", "scope": "type_a",
                "role": "analyst", "learnerId": "w15-demo-agent"},
  "typeA_read_of_A": {"status": 200, "dispute_id": "29", "dispute_type": "A"},
  "typeB_read_of_A": {"status": 403, "error": {"code": "forbidden_scope",
      "message": "Your seat (scope='type_b') is not permitted to access Type-A disputes.",
      "hint": "This access was denied and logged in the audit trail."}},
  "idempotent_replay": {"ticket_id": "9000000002", "idempotent_replay": True},
}

if LIVE:
    import requests
    K = os.environ["MERIDIAN_API_KEY"]
    me = requests.get(BASE_URL + "/me", headers={"Authorization": f"Bearer {K}"}, timeout=10).json()
    hr("LIVE /me"); print(json.dumps(me, indent=1))
    r = requests.get(BASE_URL + "/disputes/29", headers={"Authorization": f"Bearer {K}"}, timeout=10)
    print("GET /disputes/29 ->", r.status_code)
else:
    hr("recorded: the Type-A seat reads a Type-A dispute")
    print(json.dumps(RECORDED["me_type_a"], indent=1))
    print("GET /disputes/29 ->", RECORDED["typeA_read_of_A"]["status"], "(allowed)")
    hr("recorded: the Type-B seat asks the IDENTICAL question")
    print("GET /disputes/29 ->", RECORDED["typeB_read_of_A"]["status"])
    print(json.dumps(RECORDED["typeB_read_of_A"]["error"], indent=1))
    hr("recorded: the write path honors Idempotency-Key (§4's contract, in production)")
    print(json.dumps(RECORDED["idempotent_replay"], indent=1))
print("\nSame question, two identities, correctly-filtered answers + audit rows —")
print("the M3 acceptance criterion, passing in production. That row IS the renewal.")


## 9.5 Interview corner, answered — "Design a VPC-deployed HIPAA RAG over 50M documents"

The class prompt, with the model answer written out. The pass behavior is a *sequence*,
not a component list — and it maps one-to-one onto what this week built.

**1. Clarify before designing (30 seconds, out loud).** Who are the users — clinicians,
billing, researchers? What may each see — their own patients? their department's? de-identified
only? What's the update rate on the 50M docs, and is there a BAA with every vendor in the
path (model host included)? An answer that starts drawing boxes before asking these has
already failed — HIPAA design is *requirements* design.

**2. Draw the trust boundaries.** Where does PHI cross a line? Ingestion → index (PHI enters
the embedding store); query → retrieval (a user's question may itself contain PHI); retrieval
→ model context (chunks cross into the LLM); model → response (PHI exits toward a screen).
Everything stays in-VPC — the model endpoint too (a VPC-hosted/PrivateLink model, so no PHI
leaves the boundary) — encrypted in transit and at rest.

**3. Deterministic enforcement at every crossing.** At each boundary, a control that does
not depend on model behavior: document-level ACL filters *inside the retriever query*
(the index only returns chunks the acting user may see — never "retrieve then let the
prompt filter"); per-user identity carried through every hop; an output check before
display. The sentence to say verbatim: *"the model is never the access boundary."*

**4. Map it to what you've already built** — this is what makes the answer concrete
instead of recited: the governed **door** = one retrieval/tool layer all agents use (W16's
MCP server); **OBO** = every retrieval runs under the clinician's exchanged, scoped
identity, not a service account (§5); **audit** = append-only rows for every access,
denials included — which is precisely HIPAA's access-accounting requirement (§3's
`AuditLog`, grown up). Close with the differentiator: least privilege by construction,
deny-by-default, and the two-analyst test rerun as a *two-clinician test* — same question,
different patient panels, provably different answers.


## 10. Glossary — every named term from the slides

- **Reg E (Regulation E)** — the US consumer-protection rule for electronic transfers: a disputed card charge starts a 10-business-day clock to investigate, with provisional credit for most fraud-type disputes. Meridian's 12.8% breach rate against this clock IS the engagement.
- **MCP (Model Context Protocol)** — a standard plug shape (build the tool once, any agent can use it); formally the open standard for agent↔tool/data connections; servers expose tools/resources/prompts to any MCP client over JSON-RPC (stdio or HTTP).
- **Tool / Resource / Prompt (MCP)** — callable function with side effects / addressable read-only data / reusable prompt template.
- **JSON-RPC 2.0** — the wire format under MCP: `{jsonrpc, id, method, params}` in, `{result | error}` out.
- **The governed door** — the single MCP server all agent access flows through; the one place scope + audit are enforced.
- **RBAC** — role-based access control: what a *role* (Type-A analyst) may see/do, enforced per-user.
- **RLS (row-level security)** — the database itself filters rows by the caller's identity; the enforcement layer *under* the API.
- **OAuth2 on-behalf-of (OBO)** — token exchange (RFC 8693): the agent trades the user's token for one scoped to that user; no standing access.
- **Acting identity** — whose authority a call runs under (the analyst — never "the bot").
- **Least privilege / deny-by-default** — grant the minimum, refuse anything not explicitly allowed.
- **Idempotency key** — caller-supplied write ID; replays return the original instead of double-posting.
- **Structured error** — machine-readable failure (`rate_limited, retry_after=2`) an agent can reason about.
- **Retry with backoff** — re-attempt with growing delays; how tools absorb flaky backends.
- **Audit log** — append-only `{acting_identity, scope, action, resource, decision}` rows; denials included.
- **Indirect prompt injection** — attacker instructions hidden in *data* the agent retrieves (chunk 121).
- **Defense in depth** — independent layers (guardrail → scope gate → RLS) so one miss ≠ breach.
- **Two-analyst test** — identical question under two scoped identities → different, correctly-filtered answers + matching audit rows. The M3 acceptance criterion.
- **AgentCore Gateway / Identity** — managed API→MCP-tools bridge / managed OAuth token vault for agent identity.
- **A2A (Agent2Agent)** — the peer-agent wall: an open protocol for agent↔agent discovery (agent cards) and task delegation between vendors' agents; MCP connects an agent to tools, A2A connects it to *colleagues*.
- **AG-UI** — the frontend wall: a protocol for agent↔user-interface interaction — streaming state, tool progress, human-in-the-loop prompts to the analyst's screen (W17 preview).
- **Protected-resource metadata (RFC 9728)** — the `/.well-known/oauth-protected-resource` document a remote MCP server publishes so clients can discover its authorization server; part of MCP's OAuth 2.1 resource-server model (§2.1).
- **Multi-tenancy** — one deployment, many isolated tenants (per-learner keys at Meridian); tenancy walls *between* tenants (`learner_id`-namespaced rows, audit included), scope filters *within* one (§5.5).
- **Enumeration attack** — sweeping an id space and reading the *difference* between errors (`not_found` vs `403`) to learn which records exist; defeated by one denial shape for both (§3).

## Sources

- MCP spec & docs — https://modelcontextprotocol.io (Anthropic, Nov 2024; verify current spec revision)
- OAuth2 Token Exchange — RFC 8693
- Amazon Bedrock AgentCore (Gateway, Identity) — https://docs.aws.amazon.com/bedrock-agentcore/ *(verify before quoting)*
- Postgres row-level security — postgresql.org/docs/current/ddl-rowsecurity.html
- "Data is political" — Nabeel Qureshi, *Reflections on Palantir*
- The deployed proof surface — https://ik-fde-casestudy-bank.vercel.app (`/api/v1`, scope-enforced, audited)
